# BuildSite 360 — YOLOv8 PPE Safety Detection Fine-Tuning

Fine-tunes YOLOv8s to detect two classes on construction sites: **helmet** and **vest**.

**Runtime:** set to GPU before running (`Runtime > Change runtime type > T4 GPU`).

**Credentials required** — this notebook cannot run without them:
- A Kaggle API token (`kaggle.json` from https://www.kaggle.com/settings/account)
- A Roboflow API key (https://app.roboflow.com/settings/api)

Record the real metrics this notebook prints into `ai-service/TRAINING.md`. Do not copy numbers from anywhere else — the whole point of that file is to record what *this* training run actually achieved.

## 1. Environment

In [ ]:
!pip install -q ultralytics roboflow kaggle

import torch
from ultralytics import YOLO

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU. Set Runtime > Change runtime type > GPU before continuing.')

## 2. Credentials

Upload `kaggle.json` when prompted. The Roboflow key is read from a Colab secret named `ROBOFLOW_API_KEY` if present, otherwise it prompts.

In [ ]:
import os, getpass
from google.colab import files

os.makedirs('/root/.config/kaggle', exist_ok=True)
if not os.path.exists('/root/.config/kaggle/kaggle.json'):
    print('Upload your kaggle.json:')
    files.upload()
    !mv kaggle.json /root/.config/kaggle/kaggle.json
    !chmod 600 /root/.config/kaggle/kaggle.json

try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = getpass.getpass('Roboflow API key: ')

## 3. Datasets

Pull a PPE dataset from Roboflow Universe in YOLOv8 format. Roboflow emits a `data.yaml` describing train/val splits and class names.

**Check the class list it prints.** Public PPE datasets vary a lot — many ship 5–10 classes (`helmet`, `no-helmet`, `vest`, `person`, `head`, ...). The remap cell below collapses whatever it provides down to the two classes this project needs. Adjust `CLASS_MAP` to match the dataset you actually pulled.

In [ ]:
from roboflow import Roboflow
import yaml

# Replace workspace/project/version with the dataset you intend to use.
# Browse: https://universe.roboflow.com/search?q=construction+ppe
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
dataset = project.version(30).download('yolov8')

DATA_YAML = os.path.join(dataset.location, 'data.yaml')
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print('Dataset:', dataset.location)
print('Classes provided:', cfg['names'])
print('\nRecord this dataset name, version, and image count in TRAINING.md.')

In [ ]:
import glob
from pathlib import Path

# Collapse the source dataset's classes into this project's two classes.
# Keys are source class names (lowercased), values are target indices: helmet=0, vest=1.
# Labels for any class not listed here are dropped.
CLASS_MAP = {
    'helmet': 0,
    'hardhat': 0,
    'safety-helmet': 0,
    'vest': 1,
    'safety vest': 1,
    'safety-vest': 1,
}

source_names = [n.lower() for n in cfg['names']]
index_map = {i: CLASS_MAP[n] for i, n in enumerate(source_names) if n in CLASS_MAP}
print('Source index -> target index:', index_map)
print('Dropping classes:', [n for n in source_names if n not in CLASS_MAP])

kept = dropped = 0
for split in ['train', 'valid', 'test']:
    for label_path in glob.glob(f'{dataset.location}/{split}/labels/*.txt'):
        lines_out = []
        for line in Path(label_path).read_text().strip().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            src_idx = int(parts[0])
            if src_idx in index_map:
                parts[0] = str(index_map[src_idx])
                lines_out.append(' '.join(parts))
                kept += 1
            else:
                dropped += 1
        Path(label_path).write_text('\n'.join(lines_out))

print(f'Remapped {kept} boxes, dropped {dropped} boxes.')

cfg['names'] = ['helmet', 'vest']
cfg['nc'] = 2
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(cfg, f)
print('data.yaml now declares:', cfg['names'])

## 4. Training

Augmentation is tuned for outdoor/dusty site conditions:
- `hsv_h/hsv_s/hsv_v` — wide value/saturation jitter for harsh sun, overcast, and dust haze
- `degrees`, `perspective` — pole/corner-mounted cameras view workers at an angle
- `scale` — workers appear at very different distances from a fixed camera
- `fliplr` — no left/right bias on site
- `mosaic` — helps small-object detection, which matters since helmets are small at distance
- `flipud` stays 0.0: workers are upright, so vertical flips would teach nothing real

In [ ]:
model = YOLO('yolov8s.pt')  # start from COCO-pretrained weights

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,           # early-stop if val mAP plateaus
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,              # final LR = lr0 * lrf (cosine decay)
    warmup_epochs=3,
    weight_decay=0.0005,
    # --- augmentation for outdoor/dusty conditions ---
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,       # disable mosaic for last 10 epochs to stabilise
    project='runs',
    name='ppe_yolov8s',
    seed=42,
)

## 5. Evaluate on the held-out test split

In [ ]:
best_model = YOLO('runs/ppe_yolov8s/weights/best.pt')
metrics = best_model.val(data=DATA_YAML, split='test')

print('=' * 60)
print('COPY THESE NUMBERS INTO ai-service/TRAINING.md')
print('=' * 60)
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')
for i, name in enumerate(['helmet', 'vest']):
    print(f'  {name:8s} mAP@0.5 = {metrics.box.ap50[i]:.4f}')

## 6. Export weights

Download `best.pt` and place it at `ai-service/models/best.pt` in the repo.

**Expected size: roughly 20–25 MB.** A fine-tuned YOLOv8s has ~11.2M parameters, so the checkpoint lands near 22 MB — it will *not* exceed 100 MB, and a file that large would mean you trained a different architecture (YOLOv8x is ~130 MB). Size is not a proxy for quality here; judge the model by the mAP printed above.

In [ ]:
import shutil
from google.colab import files

shutil.copy('runs/ppe_yolov8s/weights/best.pt', 'best.pt')
size_mb = os.path.getsize('best.pt') / (1024 * 1024)
print(f'best.pt size: {size_mb:.1f} MB')
files.download('best.pt')